# Business health and discount dependency

Reference: **Two_Analytics_Projects_Complete_Guide.docx, Steps 1.2.1–1.2.4**.

The verified SQL is saved in `sql/02_health.sql`; results are generated by `scripts/run_business_health.py`. This notebook reads the exports so it does not interrupt DBeaver. No causal effect is estimated here.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Image, Markdown
ROOT = Path.cwd()
if not (ROOT / 'exports/business_health/weekly_kpis.csv').exists():
    ROOT = ROOT / 'outputs/grocery-marketing-effectiveness'
DATA = ROOT / 'exports/business_health'
weekly = pd.read_csv(DATA / 'weekly_kpis.csv', parse_dates=['week_start'])
category = pd.read_csv(DATA / 'category_discount.csv')
summary = pd.read_csv(DATA / 'business_health_summary.csv')
checks = pd.read_csv(DATA / 'business_health_checks.csv')
assert len(weekly) == 52 and weekly['week'].min() == 2 and weekly['week'].max() == 53
assert len(category) == 122
assert checks['passed'].astype(str).str.lower().eq('true').all()
display(summary.T)

## Weekly KPIs

Grocery purchases in source weeks 2–53. Partial week 1 is excluded. Do not sum weekly household counts to estimate annual unique households.

In [ ]:
display(weekly.head(10))
display(weekly[['households', 'sales', 'aov', 'visits_per_hh', 'discount_depth']].mean().to_frame('average_week'))
display(Image(filename=str(ROOT / 'charts/weekly_health.png')))

## Discount dependency by category

Full-period categories with at least $10,000 sales. The discounted share is weighted by shelf value, not units. Categories with no retailer-funded discounts correctly have a zero share. This is an observed share, not a forecast of lost sales without discounts.

In [ ]:
columns = ['department','product_category','sales','share_sold_on_discount','discount_depth','buyers']
display(category.sort_values('share_sold_on_discount', ascending=False)[columns].head(8))
display(category.sort_values('share_sold_on_discount')[columns].head(8))
display(Image(filename=str(ROOT / 'charts/category_discount_dependency.png')))

## Weekly association

This is an aggregate correlation. It does not control for seasonality, product mix, or campaign targeting and cannot establish a discount effect.

In [ ]:
correlation = weekly['discount_depth'].corr(weekly['sales'])
print(f'Weekly discount-depth/sales correlation: {correlation:.4f}')
display(checks)
display(Markdown((ROOT / 'docs/business_health/findings.md').read_text(encoding='utf-8')))

## Next guide section

Module 3 builds household metrics, segments, spend trajectories, and retention cohorts. The final Power BI dashboard is Step 1.7.3; these business-health exports are ready for a preview now.